# Cross-seed analysis of the Fairlearn and AIF360 experiments

This notebook reads the out-of-fold predictions written by `Fairlearn.ipynb` and `AIF360.ipynb`
(one run per random seed, in `Results/seed_<k>/<library>/`) and produces everything that depends
on more than one run:

1. **Uncertainty** (Reviewer 2; Reviewer 3, comments 2-3): mean and standard deviation over the seeds,
   95% confidence intervals from a patient-level bootstrap pooled over the seeds, and a paired
   bootstrap test of every configuration against the Original Model (Holm-corrected).
2. **Library comparison** (Reviewer 2; Reviewer 3, comment 4): both libraries' metric definitions applied
   to *identical* predictions, and a matched experiment in which Fairlearn's reduction and AIF360's
   wrapper of it are run with the same estimator, constraint, sensitive features and folds.
3. **Data bias versus model bias** (Reviewer 3, comment 5): group sizes and COVID-19 base rates in the
   data, compared with the Original Model's selection and error rates.
4. **Figures** for the paper, with 95% confidence intervals, written to `Plots/`.
**Version 6 (Reviewer 1, second round).**

5. **PCA inside the training folds** (comment 1): the notebooks fit the PCA of the image embeddings, and
   choose its dimensionality, within each training fold; the selected dimensionalities are summarised here,
   and every analysis below that refits a model uses the same folds and fold-specific PCA.
6. **Ablations and probes** (comment 2): the Original Model retrained without ethnicity, without any
   sensitive attribute, without any demographic input, and on demographics alone; and probes of how well
   sex and ethnicity can be predicted from the image features alone.
7. **Matched comparisons** (comment 3): matched folds, PCA and target attributes in the library comparison.
8. **Statistics and utility** (comment 4): direction-specific significance markers, paired comparisons of each
   sequential pipeline with its components, sensitivity, specificity, balanced accuracy, group sensitivities and
   worst-group sensitivity, and the fairness-utility trade-off.


In [1]:
import os, glob, json, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

BASE      = os.environ.get("MIDRC_RESULTS", os.path.join(os.getcwd(), "Results"))
PLOT_DIR  = os.environ.get("MIDRC_PLOTS", os.path.join(os.getcwd(), "Plots"))
N_BOOT    = int(os.environ.get("MIDRC_NBOOT", "500"))      # bootstrap replicates per seed
os.makedirs(PLOT_DIR, exist_ok=True)

SEEDS = sorted(int(p.split("_")[-1]) for p in glob.glob(os.path.join(BASE, "seed_*"))
               if os.path.exists(os.path.join(p, "AIF360", "Metrics_Values_AIF360.xlsx"))
               and os.path.exists(os.path.join(p, "Fairlearn", "Metrics_Values_Fairlear.csv")))
print("complete seeds:", SEEDS)

# configuration -> prediction file, in the order used by the figures
METHODS = {
    "Fairlearn": [
        ("Original Model", "y_pred_Original.csv", ""),
        ("Correlation Remover", "y_pred_Corr.csv", "Pre-processing"),
        ("Exponentiated Gradient", "y_pred_expGrad.csv", "In-processing"),
        ("Grid Search", "y_pred_gridsearch.csv", "In-processing"),
        ("Threshold Optimizer", "y_pred_Threshold.csv", "Post-processing"),
        ("Correlation Remover → Exponentiated Gradient", "y_pred_Corr_ExpGrad.csv", "Sequential Algorithms"),
        ("Grid Search → Threshold Optimizer", "y_pred_Grid_Threshold.csv", "Sequential Algorithms"),
        ("Correlation Remover → Threshold Optimizer", "y_pred_Corr_Threshold.csv", "Sequential Algorithms"),
    ],
    "AIF360": [
        ("Original", "y_pred_Original.csv", ""),
        ("Reweighing", "y_pred_Reweighing.csv", "Preprocessing Algorithms"),
        ("Disparate Impact Remover", "y_pred_DisparateImpactRemover.csv", "Preprocessing Algorithms"),
        ("Learning Fair Representations", "y_pred_LFR.csv", "Preprocessing Algorithms"),
        ("Exponentiated Gradient Reduction", "y_pred_EGR.csv", "In-processing Algorithms"),
        ("Grid Search Reduction", "y_pred_GSR.csv", "In-processing Algorithms"),
        ("Equalized Odds Postprocessing", "y_pred_EOP.csv", "Postprocessing Algorithms"),
        ("Calibrated Equalized Odds Postprocessing", "y_pred_CEOP.csv", "Postprocessing Algorithms"),   # Version 6
        ("Reject Option Classification", "y_pred_ROC.csv", "Postprocessing Algorithms"),
        ("RW + EGR", "y_pred_Sequential_RW_EGR.csv", "Sequential Models"),
        ("DIR + GSR", "y_pred_Sequential_DIR_GSR.csv", "Sequential Models"),
        ("EGR + CEOP", "y_pred_Sequential_EGR_CEOP.csv", "Sequential Models"),
        ("RW + CEOP", "y_pred_Sequential_RW_CEOP.csv", "Sequential Models"),
    ],
    # Version 6 (Reviewer 1, comment 2): input ablations of the Original Model (written by
    # Fairlearn.ipynb; same architecture, folds and fold-specific PCA)
    "Ablation": [
        ("Full model", "y_pred_Original.csv", "image + age + sex + ethnicity"),
        ("Image + age + sex", "y_pred_Abl_ImageAgeSex.csv", "ethnicity removed"),
        ("Image + age", "y_pred_Abl_ImageAge.csv", "no sensitive attribute"),
        ("Image only", "y_pred_Abl_ImageOnly.csv", "no demographic input"),
        ("Demographics only", "y_pred_Abl_Demographics.csv", "no image input"),
    ],
}
SOURCE_DIR = {"Fairlearn": "Fairlearn", "AIF360": "AIF360", "Ablation": "Fairlearn"}
ORIGINAL = {"Fairlearn": "Original Model", "AIF360": "Original", "Ablation": "Full model"}
# configurations whose saved predictions are probabilities (AUC can be computed)
SCORED = {("Fairlearn", "Original Model"), ("AIF360", "Original")} | {("Ablation", n) for n, _, _ in METHODS["Ablation"]}
GENDER = {0: "Female", 1: "Male"}
ETHNICITY = {0: "Asian", 1: "Black", 2: "Hispanic", 3: "Others", 4: "White"}
ATTRS = ["Gender", "Ethnicity", "Gender+Ethnicity"]


def labels_from(df):
    """Hard labels from a saved prediction file (score matrix -> argmax of the two
    Status columns; a single column is already a label vector)."""
    a = df.values
    if a.ndim == 2 and a.shape[1] == 1:
        return a[:, 0].astype(int)
    return np.argmax(a[:, :2], axis=1).astype(int)


def score_from(df):
    """P(COVID-19) from a saved probability matrix (the two Status columns, renormalised)."""
    a = df.values.astype(float)
    return a[:, 1] / np.clip(a[:, 0] + a[:, 1], 1e-12, None)


def load_seed(seed):
    out = {}
    for lib, items in METHODS.items():
        d = os.path.join(BASE, f"seed_{seed}", SOURCE_DIR[lib])
        frame = pd.read_csv(os.path.join(d, "eval_frame.csv"))
        preds, scores = {}, {}
        for name, fn, _ in items:
            p = os.path.join(d, fn)
            if os.path.exists(p):
                df = pd.read_csv(p)
                preds[name] = labels_from(df)
                if (lib, name) in SCORED:
                    scores[name] = score_from(df)
        out[lib] = (frame, preds, scores)
    return out


RUNS = {s: load_seed(s) for s in SEEDS}
frame0 = RUNS[SEEDS[0]]["Fairlearn"][0]
for s in SEEDS:                      # every run must evaluate the same images in the same order
    for lib in METHODS:
        f = RUNS[s][lib][0]
        assert (f["case_id"].values == frame0["case_id"].values).all() and (f["Status"].values == frame0["Status"].values).all()
print(len(frame0), "images |", frame0["case_id"].nunique(), "patients")
missing = [(s, lib, n) for s in SEEDS for lib, items in METHODS.items() for n, _, _ in items if n not in RUNS[s][lib][1]]
print("missing prediction files:", missing if missing else "none")


complete seeds: [0, 1, 2, 3, 4]
5999 images | 5095 patients
missing prediction files: none


## Metric definitions

Both libraries' metrics are re-implemented here in NumPy so that they can be bootstrapped quickly.
The next cell checks every AIF360 quantity against `aif360.metrics.ClassificationMetric` on real
predictions.

* **Fairlearn** (as in `Fairlearn.ipynb`): for the COVID-19 class, the demographic parity difference
  is the largest minus the smallest selection rate across groups; the equalized odds difference is the
  larger of the TPR and FPR spreads. All groups are used.
* **AIF360** (as in `AIF360.ipynb`): one group is privileged and the union of the other listed groups is
  unprivileged; each metric is the maximum over the privileged group and the favourable class. The
  ethnicity groups are Asian, Black, Hispanic and White (*Others* is not listed, as in the notebook).
  Theil and generalized entropy (α = 2) indices use the benefit b = ŷ − y + 1 over all individuals.
* **Utility** (Version 6): sensitivity (TPR), specificity and balanced accuracy for COVID-19; the TPR and FPR of every
  ethnicity and sex group; the worst-group TPR (lowest of Asian, Black, Hispanic and White); and, for the configurations
  whose outputs are probabilities (Original Model and ablations), the AUC.


In [3]:
def group_codes(frame, attr, library):
    g, e = frame["Gender"].values, frame["Ethnicity"].values
    if attr == "Gender":
        return g, [0, 1]
    if attr == "Ethnicity":
        return e, ([0, 1, 2, 3, 4] if library == "Fairlearn" else [0, 1, 2, 4])
    code_ = g * 10 + e
    eth = [0, 1, 2, 3, 4] if library == "Fairlearn" else [0, 1, 2, 4]
    return code_, [gg * 10 + ee for gg in [0, 1] for ee in eth]


def _rates(yt, yp):
    n = len(yp)
    sr = yp.mean() if n else np.nan
    pos, neg = yt == 1, yt == 0
    tpr = yp[pos].mean() if pos.any() else np.nan
    fpr = yp[neg].mean() if neg.any() else np.nan
    return sr, tpr, fpr


def _spread(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return np.nan if len(v) < 2 else v.max() - v.min()


def fairlearn_metrics(y, p, codes, groups):
    yt, yp = (y == 1).astype(int), (p == 1).astype(int)      # COVID-19 class, one-vs-rest
    r = [_rates(yt[codes == g], yp[codes == g]) for g in groups if (codes == g).any()]
    sr, tpr, fpr = map(np.array, zip(*r))
    return {"DPD": _spread(sr), "EOD": np.nanmax([_spread(tpr), _spread(fpr)])}


def _clean(v):
    return v if np.isfinite(v) else 0.0                       # same guard as the notebook


def aif360_metrics(y, p, codes, groups):
    best = dict.fromkeys(["SPD", "AOD", "EOpD", "Theil", "GEI", "DI"], -np.inf)
    listed = np.isin(codes, groups)
    for fav in (0, 1):
        yt, yp = (y == fav).astype(int), (p == fav).astype(int)
        b = yp - yt + 1.0; mu = b.mean()
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = b / mu
            theil = np.mean(np.where(ratio > 0, ratio * np.log(ratio), 0.0))
        gei = np.mean(ratio ** 2 - 1) / 2.0
        best["Theil"] = max(best["Theil"], _clean(theil)); best["GEI"] = max(best["GEI"], _clean(gei))
        for g in groups:
            priv = codes == g; unpriv = listed & ~priv
            if not priv.any() or not unpriv.any():
                continue
            srp, tprp, fprp = _rates(yt[priv], yp[priv])
            sru, tpru, fpru = _rates(yt[unpriv], yp[unpriv])
            with np.errstate(divide="ignore", invalid="ignore"):
                vals = {"SPD": sru - srp, "AOD": 0.5 * ((fpru - fprp) + (tpru - tprp)),
                        "EOpD": tpru - tprp, "DI": sru / srp}
            for k, v in vals.items():
                best[k] = max(best[k], _clean(v))
    return best


# ── Version 6 (Reviewer 1, comment 4): utility and subgroup sensitivity ──
ETH_FAIR = [0, 1, 2, 4]          # the four ethnicity groups used by the fairness metrics


def utility_metrics(y, p, g, e, score=None):
    """Overall sensitivity, specificity, balanced accuracy, group TPR/FPR, worst-group TPR
    and (for probabilistic outputs) AUC. COVID-19 is the positive class."""
    yt, yp = y == 1, p == 1
    sens = yp[yt].mean() if yt.any() else np.nan
    spec = (~yp[~yt]).mean() if (~yt).any() else np.nan
    out = {"Sensitivity": sens, "Specificity": spec, "Balanced accuracy": (sens + spec) / 2}
    tprs = []
    for codes, names in ((e, ETHNICITY), (g, GENDER)):
        for c, nm in names.items():
            m = codes == c
            out[f"TPR|{nm}"] = yp[m & yt].mean() if (m & yt).any() else np.nan
            out[f"FPR|{nm}"] = yp[m & ~yt].mean() if (m & ~yt).any() else np.nan
            if names is ETHNICITY and c in ETH_FAIR:
                tprs.append(out[f"TPR|{nm}"])
    out["Worst-group TPR"] = np.nanmin(tprs)
    out["TPR gap (ethnicity)"] = np.nanmax(tprs) - np.nanmin(tprs)
    if score is not None and 0 < yt.sum() < len(yt):
        out["AUC"] = roc_auc_score(yt, score)
    return out


def all_metrics(frame, p, library, idx=None, score=None):
    y, g, e = frame["Status"].values, frame["Gender"].values, frame["Ethnicity"].values
    if idx is not None:
        y, p, g, e = y[idx], p[idx], g[idx], e[idx]
        score = None if score is None else score[idx]
    defs = "AIF360" if library == "AIF360" else "Fairlearn"     # the ablations use Fairlearn's definitions
    out = {"Accuracy": (y == p).mean()}
    out.update(utility_metrics(y, p, g, e, score))
    for attr in ATTRS:
        codes, groups = group_codes(frame, attr, defs)
        if idx is not None:
            codes = codes[idx]
        m = fairlearn_metrics(y, p, codes, groups) if defs == "Fairlearn" else aif360_metrics(y, p, codes, groups)
        for k, v in m.items():
            out[f"{attr}|{k}"] = v
    return out


In [4]:
# ── Check the NumPy AIF360 metrics against aif360 itself (seed 0, Original model) ──
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

f, preds, _ = RUNS[SEEDS[0]]["AIF360"]
p = preds["Original"]
df = f[["Gender", "Ethnicity"]].copy(); df["Status"] = f["Status"].values
codes, groups = group_codes(f, "Ethnicity", "AIF360")
ref = dict.fromkeys(["SPD", "AOD", "EOpD", "Theil", "GEI", "DI"], -np.inf)
for fav in (0, 1):
    for gp in groups:
        ds = StandardDataset(df, label_name="Status", favorable_classes=[fav],
                             protected_attribute_names=["Ethnicity"], privileged_classes=[[gp]])
        pds = ds.copy(); pds.labels = p.astype(float).reshape(-1, 1)   # raw labels; aif360 compares with favorable_label
        gl = [{"Ethnicity": float(g)} for g in groups]
        i = groups.index(gp)
        m = ClassificationMetric(ds, pds, unprivileged_groups=gl[:i] + gl[i + 1:], privileged_groups=[gl[i]])
        for k, fn in [("SPD", m.statistical_parity_difference), ("AOD", m.average_odds_difference),
                      ("EOpD", m.equal_opportunity_difference), ("Theil", m.theil_index),
                      ("GEI", m.generalized_entropy_index), ("DI", m.disparate_impact)]:
            v = fn(); ref[k] = max(ref[k], v if np.isfinite(v) else 0.0)
mine = aif360_metrics(f["Status"].values, p, codes, groups)
check = pd.DataFrame({"aif360": ref, "numpy": mine}); check["abs diff"] = (check.aif360 - check.numpy).abs()
print(check.round(6)); assert check["abs diff"].max() < 1e-9, "metric re-implementation does not match aif360"

pip install 'aif360[inFairness]'
         aif360     numpy  abs diff
SPD    0.360863  0.360863       0.0
AOD    0.324788  0.324788       0.0
EOpD   0.361589  0.361589       0.0
Theil  0.298531  0.298531       0.0
GEI    0.213831  0.213831       0.0
DI     2.296191  2.296191       0.0


## 1. Point estimates, seed variability and bootstrap confidence intervals

In [6]:
# patient-level bootstrap: resample patients (all images of a resampled patient come along).
# The same resamples are used for every configuration of a seed, so differences are paired.
case = frame0["case_id"].values
uniq, inv = np.unique(case, return_inverse=True)
rows_of = [np.flatnonzero(inv == k) for k in range(len(uniq))]

def boot_index(rng):
    pick = rng.integers(0, len(uniq), len(uniq))
    return np.concatenate([rows_of[k] for k in pick])

records, boot = [], []          # point values per seed / bootstrap values per seed x replicate
for s in SEEDS:
    rng = np.random.default_rng(1000 + s)
    idxs = [boot_index(rng) for _ in range(N_BOOT)]
    for lib, (frame, preds, scores) in RUNS[s].items():
        for name, p in preds.items():
            sc = scores.get(name)
            records.append({"seed": s, "library": lib, "method": name, **all_metrics(frame, p, lib, score=sc)})
            for b, idx in enumerate(idxs):
                boot.append({"seed": s, "b": b, "library": lib, "method": name, **all_metrics(frame, p, lib, idx, score=sc)})
    print("seed", s, "done", flush=True)

point = pd.DataFrame(records)
bootdf = pd.DataFrame(boot)
point.to_csv(os.path.join(BASE, "point_estimates_by_seed.csv"), index=False)
# bootstrap values per configuration, in the same (seed, replicate) order for every configuration
BOOT = {k: g.sort_values(["seed", "b"]).reset_index(drop=True) for k, g in bootdf.groupby(["library", "method"], sort=False)}
print(point.groupby(["library", "method"], sort=False)[["Accuracy", "Balanced accuracy", "Sensitivity", "Specificity"]].agg(["mean", "std"]).round(4))


seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
                                                       Accuracy         Balanced accuracy         Sensitivity         Specificity        
                                                           mean     std              mean     std        mean     std        mean     std
library   method                                                                                                                         
Fairlearn Original Model                                 0.6403  0.0149            0.6403  0.0149      0.6506  0.0377      0.6301  0.0572
          Correlation Remover                            0.6143  0.0103            0.6143  0.0103      0.5811  0.0362      0.6475  0.0510
          Exponentiated Gradient                         0.5258  0.0110            0.5257  0.0110      0.2845  0.0104      0.7670  0.0191
          Grid Search                                    0.5489  0.0389            0.5489  0.0389      0.5068  0

In [7]:
# Summary: mean and SD over seeds, pooled bootstrap 95% CI, paired bootstrap test vs the original.
#
# Version 6 (Reviewer 1, comment 4): every test is two-sided and its direction is reported.
# For the disparity metrics (DPD, EOD, SPD, AOD, EOpD, Theil, GEI) a decrease is an improvement;
# for DI, accuracy, sensitivity, specificity, balanced accuracy, AUC and group TPRs an increase is.
# Holm's correction is applied within each family = one library x one metric, across all the
# configurations of that library compared with its original model (7 Fairlearn, 12 AIF360,
# 4 ablation comparisons per metric).
def holm(pvals):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj


def paired_p(diff):
    """Two-sided percentile-bootstrap p-value for a paired difference (H0: no difference)."""
    diff = np.asarray(diff, float); diff = diff[np.isfinite(diff)]
    if len(diff) == 0:
        return np.nan
    if np.all(diff == 0):
        return 1.0
    return min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean()))


def direction(d, p):
    if not np.isfinite(p) or p >= 0.05:
        return "n.s."
    return "decrease" if d < 0 else "increase"


metric_cols = [c for c in point.columns if c not in ("seed", "library", "method")]
rows = []
for lib, items in METHODS.items():
    P0 = point[(point.library == lib) & (point.method == ORIGINAL[lib])]
    for name, _, stage in items:
        P = point[(point.library == lib) & (point.method == name)]
        if P.empty:
            continue
        B, B0 = BOOT[(lib, name)], BOOT[(lib, ORIGINAL[lib])]
        for col in metric_cols:
            if P[col].isna().all():
                continue
            bv, b0 = B[col].values, B0[col].values
            diff = bv - b0
            rows.append({"library": lib, "method": name, "stage": stage, "metric": col,
                         "mean": P[col].mean(), "sd_seeds": P[col].std(ddof=1) if len(P) > 1 else np.nan,
                         "ci_low": np.nanpercentile(bv, 2.5), "ci_high": np.nanpercentile(bv, 97.5),
                         "diff_vs_original": P[col].mean() - P0[col].mean(),
                         "diff_ci_low": np.nanpercentile(diff, 2.5), "diff_ci_high": np.nanpercentile(diff, 97.5),
                         "p_vs_original": np.nan if name == ORIGINAL[lib] else paired_p(diff)})
summary = pd.DataFrame(rows)
summary["p_holm"] = np.nan
for (lib, met), g in summary.groupby(["library", "metric"]):
    ok = g["p_vs_original"].notna()
    if ok.any():
        summary.loc[g.index[ok], "p_holm"] = holm(g.loc[ok, "p_vs_original"])
summary["holm_family_size"] = summary.groupby(["library", "metric"])["p_vs_original"].transform(lambda s: s.notna().sum())
summary["significant_change"] = [direction(d, p) for d, p in zip(summary["diff_vs_original"], summary["p_holm"])]
summary.to_csv(os.path.join(BASE, "summary_with_CI.csv"), index=False)
with pd.ExcelWriter(os.path.join(BASE, "summary_with_CI.xlsx")) as xw:
    for lib in METHODS:
        summary[summary.library == lib].to_excel(xw, sheet_name=lib, index=False)
show = ["Accuracy", "Balanced accuracy", "Sensitivity", "Specificity", "Worst-group TPR"]
(summary[summary.metric.isin(show)]
 .pivot_table(index=["library", "method"], columns="metric", values="mean", sort=False)[show].round(4))


Out[5]: 
metric                                                  Accuracy  Balanced accuracy  Sensitivity  Specificity  Worst-group TPR
library   method                                                                                                              
Fairlearn Original Model                                  0.6403             0.6403       0.6506       0.6301           0.3885
          Correlation Remover                             0.6143             0.6143       0.5811       0.6475           0.5116
          Exponentiated Gradient                          0.5258             0.5257       0.2845       0.7670           0.2681
          Grid Search                                     0.5489             0.5489       0.5068       0.5910           0.4101
          Threshold Optimizer                             0.5572             0.5571       0.2886       0.8256           0.2168
          Correlation Remover → Exponentiated Gradient    0.5251             0.5251       0.2770      

In [8]:
# Discrimination (AUC) of the configurations whose outputs are probabilities: the Original Model
# of each notebook and the input ablations. Mean over seeds and pooled bootstrap 95% CI.
auc_tbl = summary[summary.metric == "AUC"][["library", "method", "mean", "sd_seeds", "ci_low", "ci_high",
                                             "diff_vs_original", "diff_ci_low", "diff_ci_high", "p_holm"]]
for s in SEEDS:
    print(f"seed {s}: AUC (Fairlearn Original Model) = "
          f"{point[(point.seed == s) & (point.library == 'Fairlearn') & (point.method == 'Original Model')]['AUC'].iloc[0]:.4f}")
auc_tbl.round(4)


seed 0: AUC (Fairlearn Original Model) = 0.6649
seed 1: AUC (Fairlearn Original Model) = 0.6985
seed 2: AUC (Fairlearn Original Model) = 0.6976
seed 3: AUC (Fairlearn Original Model) = 0.6930
seed 4: AUC (Fairlearn Original Model) = 0.7078
Out[6]: 
       library             method    mean  sd_seeds  ci_low  ci_high  diff_vs_original  diff_ci_low  diff_ci_high  p_holm
20   Fairlearn     Original Model  0.6923    0.0163  0.6556   0.7171            0.0000       0.0000        0.0000     NaN
229     AIF360           Original  0.6923    0.0163  0.6556   0.7171            0.0000       0.0000        0.0000     NaN
724   Ablation         Full model  0.6923    0.0163  0.6556   0.7171            0.0000       0.0000        0.0000     NaN
751   Ablation  Image + age + sex  0.6719    0.0099  0.6479   0.6927           -0.0205      -0.0420        0.0049  0.3536
778   Ablation        Image + age  0.6748    0.0133  0.6416   0.6959           -0.0176      -0.0303       -0.0022  0.0936
805   Ablation     

### Sequential pipelines versus their components (Version 6)

In [10]:
# Version 6 (Reviewer 1, comment 4): paired comparisons of each sequential pipeline with each of
# its components. Same pooled patient-level bootstrap resamples, two-sided paired test, Holm's
# correction within each library x metric family (6 Fairlearn and 8 AIF360 comparisons).
COMPONENTS = {
    "Fairlearn": [
        ("Correlation Remover → Exponentiated Gradient", ["Correlation Remover", "Exponentiated Gradient"]),
        ("Grid Search → Threshold Optimizer", ["Grid Search", "Threshold Optimizer"]),
        ("Correlation Remover → Threshold Optimizer", ["Correlation Remover", "Threshold Optimizer"]),
    ],
    "AIF360": [
        ("RW + EGR", ["Reweighing", "Exponentiated Gradient Reduction"]),
        ("DIR + GSR", ["Disparate Impact Remover", "Grid Search Reduction"]),
        ("EGR + CEOP", ["Exponentiated Gradient Reduction", "Calibrated Equalized Odds Postprocessing"]),
        ("RW + CEOP", ["Reweighing", "Calibrated Equalized Odds Postprocessing"]),
    ],
}
rows = []
for lib, pairs in COMPONENTS.items():
    for seq, comps in pairs:
        for comp in comps:
            if (lib, seq) not in BOOT or (lib, comp) not in BOOT:
                continue
            for col in metric_cols:
                a, b = BOOT[(lib, seq)][col].values, BOOT[(lib, comp)][col].values
                if np.isnan(a).all() or np.isnan(b).all():
                    continue
                diff = a - b
                pa = point[(point.library == lib) & (point.method == seq)][col].mean()
                pb = point[(point.library == lib) & (point.method == comp)][col].mean()
                rows.append({"library": lib, "sequential": seq, "component": comp, "metric": col,
                             "sequential_mean": pa, "component_mean": pb, "diff": pa - pb,
                             "diff_ci_low": np.nanpercentile(diff, 2.5), "diff_ci_high": np.nanpercentile(diff, 97.5),
                             "p": paired_p(diff)})
components = pd.DataFrame(rows)
components["p_holm"] = np.nan
for (lib, met), g in components.groupby(["library", "metric"]):
    ok = g["p"].notna()
    components.loc[g.index[ok], "p_holm"] = holm(g.loc[ok, "p"])
components["holm_family_size"] = components.groupby(["library", "metric"])["p"].transform(lambda s: s.notna().sum())
components["significant_change"] = [direction(d, p) for d, p in zip(components["diff"], components["p_holm"])]
components.to_csv(os.path.join(BASE, "library_component_comparisons.csv"), index=False)
key = ["Balanced accuracy", "Sensitivity", "Gender+Ethnicity|EOD", "Gender+Ethnicity|DPD",
       "Gender+Ethnicity|AOD", "Gender+Ethnicity|SPD"]
components[components.metric.isin(key)][["library", "sequential", "component", "metric", "sequential_mean",
                                         "component_mean", "diff", "diff_ci_low", "diff_ci_high", "p_holm",
                                         "significant_change"]].round(4)


Out[7]: 
       library                                    sequential                                 component                metric  sequential_mean  component_mean    diff  diff_ci_low  diff_ci_high  p_holm  \
1    Fairlearn  Correlation Remover → Exponentiated Gradient                       Correlation Remover           Sensitivity           0.2770          0.5811 -0.3041      -0.3680       -0.2467  0.0000   
3    Fairlearn  Correlation Remover → Exponentiated Gradient                       Correlation Remover     Balanced accuracy           0.5251          0.6143 -0.0892      -0.1183       -0.0567  0.0000   
24   Fairlearn  Correlation Remover → Exponentiated Gradient                       Correlation Remover  Gender+Ethnicity|DPD           0.0847          0.2835 -0.1988      -0.3092       -0.0496  0.0528   
25   Fairlearn  Correlation Remover → Exponentiated Gradient                       Correlation Remover  Gender+Ethnicity|EOD           0.1570          0.2838 -0.1268      -0.3

### Subgroup sensitivity (Version 6)

In [12]:
# Version 6 (Reviewer 1, comment 4): subgroup sensitivity. For every configuration, the TPR of
# each ethnicity and sex group, its 95% CI, the change from the original model with its CI, and
# the Holm-adjusted p-value (family = library x group, across configurations).
grp_cols = [f"TPR|{n}" for n in ETHNICITY.values()] + [f"TPR|{n}" for n in GENDER.values()] + ["Worst-group TPR"]
subgroup = summary[summary.metric.isin(grp_cols)][["library", "method", "metric", "mean", "ci_low", "ci_high",
                                                   "diff_vs_original", "diff_ci_low", "diff_ci_high", "p_holm",
                                                   "significant_change"]]
subgroup.to_csv(os.path.join(BASE, "subgroup_sensitivity.csv"), index=False)

# group sizes among COVID-19 positive images (the denominators of the group TPRs)
pos = frame0["Status"].values == 1
print("COVID-19 positive images per group:",
      {ETHNICITY[c]: int((pos & (frame0["Ethnicity"].values == c)).sum()) for c in ETHNICITY},
      {GENDER[c]: int((pos & (frame0["Gender"].values == c)).sum()) for c in GENDER})
(subgroup.assign(cell=lambda d: d["mean"].map("{:.3f}".format) + " (" + d["diff_vs_original"].map("{:+.3f}".format)
                 + (d["significant_change"].map({"decrease": " ↓", "increase": " ↑", "n.s.": ""})) + ")")
 .pivot_table(index=["library", "method"], columns="metric", values="cell", aggfunc="first", sort=False)[grp_cols])


COVID-19 positive images per group: {'Asian': 194, 'Black': 1328, 'Hispanic': 254, 'Others': 75, 'White': 1148} {'Female': 1441, 'Male': 1558}
Out[8]: 
metric                                                         TPR|Asian         TPR|Black      TPR|Hispanic        TPR|Others         TPR|White        TPR|Female          TPR|Male   Worst-group TPR
library   method                                                                                                                                                                                      
Fairlearn Original Model                                  0.823 (+0.000)    0.836 (+0.000)    0.738 (+0.000)    0.645 (+0.000)    0.389 (+0.000)    0.654 (+0.000)    0.648 (+0.000)    0.389 (+0.000)
          Correlation Remover                           0.515 (-0.307 ↓)  0.620 (-0.216 ↓)  0.617 (-0.121 ↓)    0.629 (-0.016)  0.536 (+0.148 ↑)  0.553 (-0.100 ↓)    0.607 (-0.041)    0.512 (+0.123)
          Exponentiated Gradient                    

## Fold-specific PCA (Version 6)

The notebooks' folds and fold-specific PCA are rebuilt here; the dimensionality chosen in each training fold is read from `pca_selection.csv`.

In [14]:
# Rebuild, for every seed, the notebooks' patient-grouped folds and fold-specific PCA (Version 6,
# Reviewer 1, comments 1 and 3). The folds are those of grouped_cv() (StratifiedGroupKFold, 5 folds,
# shuffled with the seed); the number of components k of each fold is read from the notebooks'
# pca_selection.csv, and the PCA is fitted on the training fold only.
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold

meta = pd.read_csv(os.environ.get("MIDRC_DATA_CSV", r"C:\midrc\dataset_final.csv"))
feat = np.load(os.environ.get("MIDRC_FEATURES", r"C:\midrc\features_xrv_densenet121.npz"), allow_pickle=True)
emb = pd.DataFrame(feat["embeddings"], index=feat["fname"])
meta = meta[meta["fname"].isin(emb.index)].reset_index(drop=True)
if os.environ.get("MIDRC_SMOKE"):
    meta = meta.sample(int(os.environ["MIDRC_SMOKE"]), random_state=42).reset_index(drop=True)
assert (meta["case_id"].values == frame0["case_id"].values).all()
EMB = emb.loc[meta["fname"]].values.astype("float64")
Y0, G0 = frame0["Status"].values, frame0["case_id"].values

pca_sel = pd.concat([pd.read_csv(os.path.join(BASE, f"seed_{s}", "Fairlearn", "pca_selection.csv")).assign(seed=s)
                     for s in SEEDS], ignore_index=True)
FOLD_CACHE = {}


def seed_folds(seed):
    """[(train, test, Z)] for the seed: Z = PCA scores of all images, fitted on the training fold,
    components beyond the fold's k set to zero (as in the notebooks)."""
    if seed in FOLD_CACHE:
        return FOLD_CACHE[seed]
    ks = pca_sel[pca_sel.seed == seed].sort_values("fold")
    out = []
    split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(EMB, Y0, G0)
    for (tr, te), (_, r) in zip(split, ks.iterrows()):
        assert len(tr) == r["n_train"], "fold reconstruction does not match the notebook"
        Z = PCA(64, random_state=seed).fit(EMB[tr]).transform(EMB)
        Z[:, int(r["k"]):] = 0.0
        out.append((tr, te, Z, int(r["k"])))
    FOLD_CACHE[seed] = out
    return out


# PCA dimensionality chosen inside the training folds
pca_summary = (pca_sel.groupby("k").size().rename("folds").to_frame()
               .assign(share=lambda d: d["folds"] / d["folds"].sum()))
pca_sel.to_csv(os.path.join(BASE, "pca_selection_all_seeds.csv"), index=False)
display(pca_sel.round(4)); pca_summary


    fold  n_train   k  inner_auc_k8  inner_auc_k16  inner_auc_k32  inner_auc_k64  seed
0      1     4783  32        0.6566         0.6725         0.6872         0.6890     0
1      2     4779  32        0.6398         0.6637         0.6830         0.6870     0
2      3     4824  64        0.6578         0.6712         0.6894         0.6970     0
3      4     4829  64        0.6531         0.6689         0.6870         0.6941     0
4      5     4781  64        0.6490         0.6641         0.6874         0.6976     0
5      1     4804  64        0.6489         0.6672         0.6828         0.6924     1
6      2     4779  32        0.6449         0.6656         0.6849         0.6886     1
7      3     4807  64        0.6488         0.6637         0.6824         0.6926     1
8      4     4813  32        0.6557         0.6696         0.6898         0.6897     1
9      5     4793  64        0.6530         0.6691         0.6870         0.6947     1
10     1     4780  32        0.6465        

## Input ablations and demographic probes (Version 6)

In [16]:
# Version 6 (Reviewer 1, comment 2): sensitive-attribute and demographic-only ablations, and
# probes of how much demographic information the image features carry.
#
# (a) Ablations: the Original Model architecture retrained on subsets of its inputs (same folds,
#     same fold-specific PCA). Utility, AUC and the Fairlearn disparity metrics, each with its
#     95% CI and a paired test against the full model (Holm within metric, 4 comparisons).
abl_cols = ["AUC", "Accuracy", "Balanced accuracy", "Sensitivity", "Specificity", "Worst-group TPR",
            "Gender|DPD", "Gender|EOD", "Ethnicity|DPD", "Ethnicity|EOD", "Gender+Ethnicity|DPD", "Gender+Ethnicity|EOD"]
ablation = summary[(summary.library == "Ablation") & summary.metric.isin(abl_cols)]
ablation.to_csv(os.path.join(BASE, "ablation_summary.csv"), index=False)
display(ablation.assign(cell=lambda d: d["mean"].map("{:.3f}".format) + " [" + d["ci_low"].map("{:.3f}".format) + ", "
                        + d["ci_high"].map("{:.3f}".format) + "]"
                        + d["significant_change"].map({"decrease": " ↓", "increase": " ↑", "n.s.": ""}))
        .pivot_table(index="method", columns="metric", values="cell", aggfunc="first", sort=False)[abl_cols].T)

# (b) Probes: can sex and ethnicity be predicted from the image features alone? A logistic
#     regression on the fold-specific PCA features (training fold only), evaluated out of fold.
#     High probe AUC means the image features themselves encode the attribute, so removing the
#     attribute columns from the input cannot remove it.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

gsex, geth = frame0["Gender"].values, frame0["Ethnicity"].values
probe_rows = []
for s in SEEDS:
    p_sex = np.zeros(len(gsex)); p_eth = np.zeros((len(geth), 5))
    for tr, te, Z, k in seed_folds(s):
        F = Z[:, :k]
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(F[tr], gsex[tr])
        p_sex[te] = m.predict_proba(F[te])[:, 1]
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(F[tr], geth[tr])
        p_eth[np.ix_(te, m.classes_.astype(int))] = m.predict_proba(F[te])
    r = {"seed": s, "Sex (Male vs Female)": roc_auc_score(gsex, p_sex)}
    for c, nm in ETHNICITY.items():
        r[f"Ethnicity: {nm} vs rest"] = roc_auc_score(geth == c, p_eth[:, c]) if 0 < (geth == c).sum() < len(geth) else np.nan
    r["Ethnicity (macro one-vs-rest)"] = np.nanmean([r[f"Ethnicity: {nm} vs rest"] for nm in ETHNICITY.values()])
    probe_rows.append(r)
probes = pd.DataFrame(probe_rows)
probe_summary = probes.drop(columns="seed").agg(["mean", "std"]).T
probes.to_csv(os.path.join(BASE, "demographic_probes_by_seed.csv"), index=False)
probe_summary.to_csv(os.path.join(BASE, "demographic_probes.csv"))
probe_summary.round(4)


method                          Full model       Image + age + sex             Image + age              Image only       Demographics only
metric                                                                                                                                    
AUC                   0.692 [0.656, 0.717]    0.672 [0.648, 0.693]    0.675 [0.642, 0.696]    0.687 [0.671, 0.703]  0.594 [0.545, 0.627] ↓
Accuracy              0.640 [0.606, 0.661]    0.626 [0.602, 0.645]    0.625 [0.597, 0.645]    0.638 [0.624, 0.653]  0.577 [0.518, 0.620] ↓
Balanced accuracy     0.640 [0.606, 0.661]    0.626 [0.602, 0.645]    0.625 [0.597, 0.645]    0.638 [0.624, 0.653]  0.577 [0.519, 0.620] ↓
Sensitivity           0.651 [0.594, 0.715]    0.600 [0.565, 0.638]    0.630 [0.598, 0.662]  0.594 [0.559, 0.638] ↓  0.520 [0.327, 0.597] ↓
Specificity           0.630 [0.548, 0.701]    0.651 [0.624, 0.681]    0.620 [0.581, 0.673]    0.682 [0.630, 0.723]    0.634 [0.555, 0.726]
Worst-group TPR       0.389

## 2. Why do the two libraries disagree?

### 2a. Same predictions, both metric definitions

In [18]:
# For each pair of corresponding configurations, apply BOTH libraries' metric definitions
# to the SAME predictions. Differences within a row come from the metric definition;
# differences between the two rows of a pair come from the algorithm/implementation.
PAIRS = [("Original Model", "Original"), ("Exponentiated Gradient", "Exponentiated Gradient Reduction"),
         ("Grid Search", "Grid Search Reduction")]
rows = []
for s in SEEDS:
    for fl_name, aif_name in PAIRS:
        for lib, name in [("Fairlearn", fl_name), ("AIF360", aif_name)]:
            frame, preds, _ = RUNS[s][lib]
            if name not in preds:
                continue
            p = preds[name]
            for attr in ATTRS:
                cf, gf = group_codes(frame, attr, "Fairlearn"); ca, ga = group_codes(frame, attr, "AIF360")
                fm = fairlearn_metrics(frame["Status"].values, p, cf, gf)
                am = aif360_metrics(frame["Status"].values, p, ca, ga)
                rows.append({"seed": s, "pair": f"{fl_name} / {aif_name}", "predictions from": f"{lib}: {name}",
                             "attribute": attr, "Accuracy": (frame["Status"].values == p).mean(),
                             "Fairlearn DPD": fm["DPD"], "Fairlearn EOD": fm["EOD"],
                             "AIF360 SPD": am["SPD"], "AIF360 AOD": am["AOD"], "AIF360 EOpD": am["EOpD"]})
same_pred = (pd.DataFrame(rows).groupby(["pair", "predictions from", "attribute"], sort=False)
             .mean(numeric_only=True).drop(columns="seed"))
same_pred.to_csv(os.path.join(BASE, "library_same_predictions.csv"))
same_pred.round(4)

Out[11]: 
                                                                                                              Accuracy  Fairlearn DPD  Fairlearn EOD  AIF360 SPD  AIF360 AOD  AIF360 EOpD
pair                                               predictions from                         attribute                                                                                    
Original Model / Original                          Fairlearn: Original Model                Gender              0.6403         0.0131         0.0235      0.0131      0.0130       0.0235
                                                                                            Ethnicity           0.6403         0.4795         0.4536      0.4576      0.4150       0.4317
                                                                                            Gender+Ethnicity    0.6403         0.5100         0.4906      0.3418      0.3062       0.3507
                                                   AIF360: O

### 2b. Matched experiment: same reduction, same settings, same folds, both libraries

AIF360's `ExponentiatedGradientReduction` and `GridSearchReduction` call Fairlearn's reductions
internally. Here both are run, for every seed, with an identical estimator (logistic regression), the same
constraint (equalized odds), the same sensitive features (sex and ethnicity jointly; AIF360 keeps numeric
protected attributes as they are, so both see the same ten groups), the same budget, the notebooks'
patient-grouped folds and the notebooks' fold-specific PCA features, so any remaining difference is due to
the implementation alone. The table above lists the settings of the two notebooks, which are now matched
in folds, PCA, target attributes and calibration data.


In [20]:
import random as _random
from sklearn.linear_model import LogisticRegression
from fairlearn.reductions import ExponentiatedGradient, GridSearch, EqualizedOdds
from aif360.datasets import StandardDataset
from aif360.algorithms.inprocessing import ExponentiatedGradientReduction, GridSearchReduction

# configuration of the two notebooks (Version 6: folds, PCA and target attributes now matched)
config = pd.DataFrame([
    ["Folds", "5 patient-grouped folds per seed", "the same 5 folds (same splitter and seed)"],
    ["PCA of the image embeddings", "fitted in each training fold; k by inner CV", "the same fold-specific PCA"],
    ["Sensitive features used by the reductions", "sex and ethnicity (joint)", "sex and ethnicity (joint)"],
    ["Pre-processing target", "Correlation Remover: sex and ethnicity",
     "RW, DIR, LFR: the joint sex x ethnicity groups"],
    ["Post-processing target", "Threshold Optimizer: sex x ethnicity, equalized odds",
     "EOP, CEOP, ROC: White vs all other patients (binary, as the algorithms require)"],
    ["Post-processing fitted on", "25% calibration part of the training fold", "the same calibration part"],
    ["Grid Search size", "40", "5 (GSR_GRID_SIZE)"],
    ["Exponentiated Gradient iterations", "10", "10 (EGR_MAX_ITER)"],
    ["Binary task handling", "one model per Status class (one-vs-rest)", "one model per Status class (one-vs-rest)"],
    ["Fairness metric groups (ethnicity)", "all 5 groups, max - min", "Asian, Black, Hispanic, White; one vs rest"],
    ["Fairness metric aggregation", "spread across groups", "max over privileged group and favourable class"],
], columns=["Setting", "Fairlearn notebook", "AIF360 notebook"])
config.to_csv(os.path.join(BASE, "library_configuration.csv"), index=False)
display(config)

# Matched experiment (Reviewer 1, comment 3): for every seed, Fairlearn's reductions and AIF360's
# wrappers of them are run with the same estimator (logistic regression), constraint (equalized
# odds), budget, sensitive features (sex and ethnicity, joint), the notebooks' folds and the
# notebooks' fold-specific PCA features. Any remaining difference is due to the implementation.
def lr():
    return LogisticRegression(max_iter=2000)


def ds(Xp, yp):
    # numeric protected attributes are kept as they are by AIF360 (no binarisation), so the
    # reductions see the same joint groups as Fairlearn's
    return StandardDataset(pd.concat([Xp, pd.Series(yp, name="Status")], axis=1), label_name="Status",
                           favorable_classes=[1], protected_attribute_names=["Gender", "Ethnicity"],
                           privileged_classes=[[1], [4]])


RUN_NAMES = ["FL ExpGrad", "AIF EGR", "FL GridSearch", "AIF GSR"]
rows = []
for s in SEEDS:
    out = {k: np.zeros(len(Y0), int) for k in RUN_NAMES}
    for tr, te, Z, k in seed_folds(s):
        Xm = frame0[["Age", "Gender", "Ethnicity"]].copy().astype(float)
        Xm[[f"PCA{i}" for i in range(k)]] = Z[:, :k]
        SF = Xm["Gender"].astype(int).astype(str) + "|" + Xm["Ethnicity"].astype(int).astype(str)
        Xtr, Xte = Xm.iloc[tr].reset_index(drop=True), Xm.iloc[te].reset_index(drop=True)
        ytr = pd.Series(Y0[tr], name="Status")
        np.random.seed(s); _random.seed(s)
        eg = ExponentiatedGradient(lr(), constraints=EqualizedOdds(), eps=0.01, max_iter=50)
        eg.fit(Xtr, ytr, sensitive_features=SF.iloc[tr].values)
        out["FL ExpGrad"][te] = eg.predict(Xte, random_state=s)
        gs = GridSearch(lr(), constraints=EqualizedOdds(), grid_size=20)
        gs.fit(Xtr, ytr, sensitive_features=SF.iloc[tr].values)
        out["FL GridSearch"][te] = gs.predict(Xte)
        np.random.seed(s); _random.seed(s)
        egr = ExponentiatedGradientReduction(estimator=lr(), constraints="EqualizedOdds", eps=0.01, max_iter=50,
                                             drop_prot_attr=False)
        egr.fit(ds(Xtr, ytr.values))
        out["AIF EGR"][te] = egr.predict(ds(Xte, Y0[te])).labels.ravel().astype(int)
        gsr = GridSearchReduction(estimator=lr(), constraints="EqualizedOdds", grid_size=20, drop_prot_attr=False)
        gsr.fit(ds(Xtr, ytr.values))
        out["AIF GSR"][te] = gsr.predict(ds(Xte, Y0[te])).labels.ravel().astype(int)
    for a, b in [("FL ExpGrad", "AIF EGR"), ("FL GridSearch", "AIF GSR")]:
        for name in (a, b):
            r = {"seed": s, "run": name, "agreement with partner": (out[a] == out[b]).mean()}
            r.update(all_metrics(frame0, out[name], "Fairlearn"))
            rows.append(r)
    print("seed", s, "done", flush=True)
matched_seeds = pd.DataFrame(rows)
matched_seeds.to_csv(os.path.join(BASE, "library_matched_experiment_by_seed.csv"), index=False)
mcols = ["agreement with partner", "Accuracy", "Balanced accuracy", "Gender|EOD", "Ethnicity|EOD",
         "Gender+Ethnicity|DPD", "Gender+Ethnicity|EOD"]
matched = matched_seeds.groupby("run", sort=False)[mcols].agg(["mean", "std"])
matched.to_csv(os.path.join(BASE, "library_matched_experiment.csv"))
matched.round(4)


                                      Setting                                 Fairlearn notebook                                    AIF360 notebook
0                                       Folds                   5 patient-grouped folds per seed          the same 5 folds (same splitter and seed)
1                 PCA of the image embeddings        fitted in each training fold; k by inner CV                         the same fold-specific PCA
2   Sensitive features used by the reductions                          sex and ethnicity (joint)                          sex and ethnicity (joint)
3                       Pre-processing target             Correlation Remover: sex and ethnicity     RW, DIR, LFR: the joint sex x ethnicity groups
4                      Post-processing target  Threshold Optimizer: sex x ethnicity, equalize...  EOP, CEOP, ROC: White vs all other patients (b...
5                   Post-processing fitted on          25% calibration part of the training fold                

## 3. Data bias versus model bias

In [22]:
f, preds, _ = RUNS[SEEDS[0]]["Fairlearn"]
y = f["Status"].values
rows = []
for attr in ["Gender", "Ethnicity"]:
    codes = f[attr].values; names = GENDER if attr == "Gender" else ETHNICITY
    for g in sorted(np.unique(codes)):
        m = codes == g
        sel = np.mean([RUNS[s]["Fairlearn"][1]["Original Model"][m].mean() for s in SEEDS])
        tpr = np.mean([RUNS[s]["Fairlearn"][1]["Original Model"][m & (y == 1)].mean() for s in SEEDS])
        fpr = np.mean([RUNS[s]["Fairlearn"][1]["Original Model"][m & (y == 0)].mean() for s in SEEDS])
        rows.append({"attribute": attr, "group": names[g], "n": m.sum(), "share": m.mean(),
                     "COVID-19 base rate (data)": y[m].mean(), "predicted COVID-19 rate (model)": sel,
                     "TPR": tpr, "FPR": fpr})
groups_tbl = pd.DataFrame(rows)
groups_tbl.to_csv(os.path.join(BASE, "data_vs_model_by_group.csv"), index=False)

summ = []
for attr in ["Gender", "Ethnicity"]:
    t = groups_tbl[groups_tbl.attribute == attr]
    data_gap = t["COVID-19 base rate (data)"].max() - t["COVID-19 base rate (data)"].min()
    model_gap = t["predicted COVID-19 rate (model)"].max() - t["predicted COVID-19 rate (model)"].min()
    summ.append({"attribute": attr, "base-rate gap in the data": data_gap,
                 "selection-rate gap of the model (DPD)": model_gap,
                 "model gap / data gap": model_gap / data_gap if data_gap else np.nan,
                 "corr(base rate, predicted rate) across groups":
                     np.corrcoef(t["COVID-19 base rate (data)"], t["predicted COVID-19 rate (model)"])[0, 1]})
data_bias = pd.DataFrame(summ)
data_bias.to_csv(os.path.join(BASE, "data_vs_model_summary.csv"), index=False)
display(groups_tbl.round(3)); data_bias.round(3)

   attribute     group     n  share  COVID-19 base rate (data)  predicted COVID-19 rate (model)    TPR    FPR
0     Gender    Female  2880  0.480                      0.500                            0.514  0.654  0.374
1     Gender      Male  3119  0.520                      0.500                            0.507  0.648  0.366
2  Ethnicity     Asian   415  0.069                      0.467                            0.686  0.823  0.566
3  Ethnicity     Black  2082  0.347                      0.638                            0.767  0.836  0.646
4  Ethnicity  Hispanic   368  0.061                      0.690                            0.688  0.738  0.577
5  Ethnicity    Others   148  0.025                      0.507                            0.455  0.645  0.260
6  Ethnicity     White  2986  0.498                      0.384                            0.288  0.389  0.224
Out[13]: 
   attribute  base-rate gap in the data  selection-rate gap of the model (DPD)  model gap / data gap  corr(bas

In [23]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [2, 5]})
for ax, attr in zip(axes, ["Gender", "Ethnicity"]):
    t = groups_tbl[groups_tbl.attribute == attr]
    x = np.arange(len(t)); w = 0.38
    ax.bar(x - w / 2, 100 * t["COVID-19 base rate (data)"], w, color="#7b9cda", edgecolor="black", hatch="///", label="COVID-19 base rate in the data")
    ax.bar(x + w / 2, 100 * t["predicted COVID-19 rate (model)"], w, color="#f4845f", edgecolor="black", hatch="xxx", label="Predicted COVID-19 rate (Original Model)")
    for xi, n in zip(x, t["n"]):
        ax.text(xi, 2, f"n={n}", ha="center", fontsize=8, color="white", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(t["group"]); ax.set_ylim(0, 100); ax.set_title(attr)
    ax.set_ylabel("Rate (%)"); ax.grid(axis="y", linestyle="--", alpha=0.4)
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Data bias versus model bias", fontsize=13)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Data-vs-Model-Bias.png"), dpi=150, bbox_inches="tight"); plt.show()

## 4. Figures (mean over seeds, 95% bootstrap CI)

In [25]:
COLORS = ["#2ec4b6", "#f4845f", "#7b9cda", "#d966a0", "#8bc34a"]
HATCHES = ["///", "///", "xxx", "...", "///"]
import textwrap

def get(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return (r["mean"].iloc[0], r["ci_low"].iloc[0], r["ci_high"].iloc[0], r["p_holm"].iloc[0]) if len(r) else (np.nan,) * 4

# Version 6 (Reviewer 1, comment 4): direction-specific markers. ↓ = significant decrease, ↑ = significant
# increase relative to the original model (two-sided paired patient-level bootstrap, Holm-adjusted
# p < 0.05 within the library x metric family). For the disparity metrics plotted here ↓ is an improvement.
def mark(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return {"decrease": "↓", "increase": "↑"}.get(r["significant_change"].iloc[0], "") if len(r) else ""

def plot(lib, cols, labels, title, fname, ylabel="Metric value (%)"):
    items = [(n, st) for n, _, st in METHODS[lib] if not summary[(summary.library == lib) & (summary.method == n)].empty]
    names = [n for n, _ in items]; cats = [st for _, st in items]
    n, k = len(names), len(cols)
    bw = min(0.8 / k, 0.22); x = np.arange(n); off = (np.arange(k) - (k - 1) / 2) * bw
    fig, ax1 = plt.subplots(figsize=(max(12, 1.7 * n), 7.5)); ax2 = ax1.twinx()
    top = 0
    for j, (col, lab) in enumerate(zip(cols, labels)):
        v = np.array([get(lib, nm, col) for nm in names], float)
        m, lo, hi, p = v[:, 0] * 100, v[:, 1] * 100, v[:, 2] * 100, v[:, 3]
        ax1.bar(x + off[j], m, bw, color=COLORS[j % 5], hatch=HATCHES[j % 5], edgecolor="black", linewidth=0.6, label=lab, zorder=3)
        ax1.errorbar(x + off[j], m, yerr=np.clip(np.array([m - lo, hi - m], dtype=float), 0, None), fmt="none", ecolor="black", elinewidth=0.9, capsize=2.5, zorder=4)
        for xi, mi, hii, nm in zip(x + off[j], m, hi, names):
            ax1.text(xi, hii + 0.6, f"{mi:.1f}{mark(lib, nm, col)}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        top = max(top, np.nanmax(hi))
    acc = np.array([get(lib, nm, "Accuracy") for nm in names], float) * 100
    ax2.errorbar(x, acc[:, 0], yerr=np.clip(np.array([acc[:, 0] - acc[:, 1], acc[:, 2] - acc[:, 0]], dtype=float), 0, None), color="black", marker="o", ms=6, lw=1.6, capsize=3, zorder=5)
    for xi, a in zip(x, acc[:, 0]):
        ax2.text(xi, a + 2, f"{a:.1f}%", ha="center", fontsize=8.5, fontweight="bold")
    ax1.set_ylim(0, max(10, np.ceil(top * 1.35 / 5) * 5)); ax2.set_ylim(0, 100)
    ax1.set_xlim(-0.5, n - 0.5); ax1.set_xticks(x); ax1.set_xticklabels([textwrap.fill(s, 16) for s in names], fontsize=8.5)
    ax1.set_ylabel(ylabel); ax2.set_ylabel("Accuracy (%)"); ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    start = 0
    for i in range(1, n + 1):
        if i == n or cats[i] != cats[start]:
            if cats[start]:
                ax1.text((start + i - 1) / 2, -0.17, cats[start], transform=ax1.get_xaxis_transform(), ha="center", va="top", fontsize=9, style="italic", color="#444")
            if i < n:
                ax1.axvline(i - 0.5, color="#999", ls=":", lw=1)
            start = i
    handles = [mpatches.Patch(facecolor=COLORS[j % 5], hatch=HATCHES[j % 5], edgecolor="black", label=l) for j, l in enumerate(labels)]
    handles.append(plt.Line2D([0], [0], color="black", marker="o", label="Accuracy"))
    ax1.legend(handles=handles, loc="upper left", fontsize=8.5, framealpha=0.9)
    ax1.set_title(title + f"  (mean of {len(SEEDS)} seeds, 95% CI; ↓/↑ significant decrease/increase vs. original, Holm)", fontsize=11)
    fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, fname), dpi=150, bbox_inches="tight"); plt.show()

for attr, tag in [("Gender", "Gender"), ("Ethnicity", "Ethnicity"), ("Gender+Ethnicity", "Gender-Ethnicity")]:
    plot("Fairlearn", [f"{attr}|DPD"], ["Demographic Parity Difference (COVID-19)"], f"Fairlearn - {attr}",
         f"Fairlearn-{tag}-Demographic_Parity.png", "Demographic Parity (%)")
    plot("Fairlearn", [f"{attr}|EOD"], ["Equalized Odds Difference (COVID-19)"], f"Fairlearn - {attr}",
         f"Fairlearn-{tag}-Equalized_Odds.png", "Equalized Odds (%)")
    plot("AIF360", [f"{attr}|{m}" for m in ["SPD", "AOD", "EOpD", "Theil", "GEI"]],
         ["Statistical Parity Difference", "Average Odds Difference", "Equal Opportunity Difference",
          "Theil Index", "Generalized Entropy Index"], f"AIF360 - {attr}", f"AIF360-{tag}.png", "Metrics Values (%)")
print(sorted(os.listdir(PLOT_DIR)))

['AIF360-Ethnicity.png', 'AIF360-Gender-Ethnicity.png', 'AIF360-Gender.png', 'Data-vs-Model-Bias.png', 'Fairlearn-Ethnicity-Demographic_Parity.png', 'Fairlearn-Ethnicity-Equalized_Odds.png', 'Fairlearn-Gender-Demographic_Parity.png', 'Fairlearn-Gender-Equalized_Odds.png', 'Fairlearn-Gender-Ethnicity-Demographic_Parity.png', 'Fairlearn-Gender-Ethnicity-Equalized_Odds.png']


### Version 6 figures: subgroup sensitivity, fairness-utility trade-off, ablations

In [27]:
# Version 6 figures (Reviewer 1, comments 2 and 4)
SHORT = {"Original Model": "Original", "Correlation Remover": "CR", "Exponentiated Gradient": "EG", "Grid Search": "GS",
         "Threshold Optimizer": "TO", "Correlation Remover → Exponentiated Gradient": "CR→EG",
         "Grid Search → Threshold Optimizer": "GS→TO", "Correlation Remover → Threshold Optimizer": "CR→TO",
         "Original": "Original", "Reweighing": "RW", "Disparate Impact Remover": "DIR",
         "Learning Fair Representations": "LFR", "Exponentiated Gradient Reduction": "EGR",
         "Grid Search Reduction": "GSR", "Equalized Odds Postprocessing": "EOP",
         "Calibrated Equalized Odds Postprocessing": "CEOP", "Reject Option Classification": "ROC",
         "RW + EGR": "RW+EGR", "DIR + GSR": "DIR+GSR", "EGR + CEOP": "EGR+CEOP", "RW + CEOP": "RW+CEOP"}
STAGE_COLOR = {"": "#333333", "Pre-processing": "#2ec4b6", "Preprocessing Algorithms": "#2ec4b6",
               "In-processing": "#7b9cda", "In-processing Algorithms": "#7b9cda",
               "Post-processing": "#f4845f", "Postprocessing Algorithms": "#f4845f",
               "Sequential Algorithms": "#d966a0", "Sequential Models": "#d966a0"}


def val(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return (r["mean"].iloc[0], r["ci_low"].iloc[0], r["ci_high"].iloc[0]) if len(r) else (np.nan,) * 3


# (1) Subgroup sensitivity: TPR of each ethnicity group, per configuration, with 95% CIs
ETH_SHOW = [(0, "Asian", "#2ec4b6", "o"), (1, "Black", "#f4845f", "s"), (2, "Hispanic", "#7b9cda", "^"), (4, "White", "#d966a0", "D")]
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
for ax, lib in zip(axes, ["Fairlearn", "AIF360"]):
    names = [n for n, _, _ in METHODS[lib] if (lib, n) in BOOT]
    x = np.arange(len(names))
    for j, (c, nm, colr, mk) in enumerate(ETH_SHOW):
        v = np.array([val(lib, n, f"TPR|{nm}") for n in names]) * 100
        xx = x + (j - 1.5) * 0.13
        ax.errorbar(xx, v[:, 0], yerr=np.clip(np.array([v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]], dtype=float), 0, None), fmt=mk, color=colr, ms=5, capsize=2, lw=1, label=nm)
        for xi, n, yi in zip(xx, names, v[:, 0]):
            r = summary[(summary.library == lib) & (summary.method == n) & (summary.metric == f"TPR|{nm}")]
            tag = {"decrease": "↓", "increase": "↑"}.get(r["significant_change"].iloc[0], "") if len(r) else ""
            if tag:
                ax.text(xi, yi + 3, tag, ha="center", fontsize=9, color=colr, fontweight="bold")
    o = np.array([val(lib, n, "Sensitivity") for n in names]) * 100
    ax.plot(x, o[:, 0], "k_", ms=22, mew=2, label="All patients")
    ax.set_xticks(x); ax.set_xticklabels([SHORT.get(n, n) for n in names], fontsize=9)
    ax.set_ylim(0, 105); ax.set_ylabel("Sensitivity (TPR, %)"); ax.grid(axis="y", ls="--", alpha=0.4)
    ax.set_title(f"{lib}: COVID-19 sensitivity by ethnicity (mean of {len(SEEDS)} seeds, 95% CI; "
                 "↓/↑ significant change vs. original, Holm)", fontsize=11)
axes[0].legend(ncol=5, fontsize=9, loc="lower left")
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Subgroup-Sensitivity.png"), dpi=150, bbox_inches="tight"); plt.show()

# (2) Fairness-utility trade-off: balanced accuracy against the joint sex x ethnicity disparity
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, lib, met, lab in [(axes[0], "Fairlearn", "Gender+Ethnicity|EOD", "Equalized odds difference, sex x ethnicity (%)"),
                          (axes[1], "AIF360", "Gender+Ethnicity|AOD", "Average odds difference, sex x ethnicity (%)")]:
    for n, _, st in METHODS[lib]:
        if (lib, n) not in BOOT:
            continue
        fx, fl, fh = np.array(val(lib, n, met)) * 100
        uy, ul, uh = np.array(val(lib, n, "Balanced accuracy")) * 100
        ax.errorbar(fx, uy, xerr=np.clip(np.array([[fx - fl], [fh - fx]], dtype=float), 0, None),
                    yerr=np.clip(np.array([[uy - ul], [uh - uy]], dtype=float), 0, None), fmt="o", ms=7,
                    color=STAGE_COLOR.get(st, "#333"), capsize=2, lw=1)
        ax.annotate(SHORT.get(n, n), (fx, uy), textcoords="offset points", xytext=(5, 5), fontsize=8.5)
    ax.set_xlabel(lab); ax.set_ylabel("Balanced accuracy (%)"); ax.grid(ls="--", alpha=0.4); ax.set_title(lib)
handles = [mpatches.Patch(color=c, label=l) for l, c in [("Original", "#333333"), ("Pre-processing", "#2ec4b6"),
           ("In-processing", "#7b9cda"), ("Post-processing", "#f4845f"), ("Sequential", "#d966a0")]]
axes[1].legend(handles=handles, fontsize=9, loc="best")
fig.suptitle(f"Fairness-utility trade-off (mean of {len(SEEDS)} seeds, 95% CI; lower left of each panel = fairer, upper = more useful)", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Fairness-Utility.png"), dpi=150, bbox_inches="tight"); plt.show()

# (3) Ablations: AUC, balanced accuracy and disparity of the full model and its input ablations
names = [n for n, _, _ in METHODS["Ablation"] if ("Ablation", n) in BOOT]
panels = [("AUC", "AUC (%)"), ("Balanced accuracy", "Balanced accuracy (%)"),
          ("Ethnicity|EOD", "Equalized odds difference, ethnicity (%)"), ("Gender|EOD", "Equalized odds difference, sex (%)")]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (met, lab) in zip(axes, panels):
    v = np.array([val("Ablation", n, met) for n in names]) * 100
    cols = ["#333333", "#7b9cda", "#2ec4b6", "#8bc34a", "#f4845f"][:len(names)]
    ax.bar(np.arange(len(names)), v[:, 0], color=cols, edgecolor="black", yerr=np.clip(np.array([v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]], dtype=float), 0, None), capsize=3)
    for xi, n, yi, hi in zip(np.arange(len(names)), names, v[:, 0], v[:, 2]):
        r = summary[(summary.library == "Ablation") & (summary.method == n) & (summary.metric == met)]
        tag = {"decrease": "↓", "increase": "↑"}.get(r["significant_change"].iloc[0], "") if len(r) else ""
        ax.text(xi, hi + 1, f"{yi:.1f}{tag}", ha="center", fontsize=8.5)
    ax.set_xticks(np.arange(len(names))); ax.set_xticklabels([textwrap.fill(n, 10) for n in names], fontsize=8.5)
    ax.set_title(lab, fontsize=10); ax.grid(axis="y", ls="--", alpha=0.4)
    if met in ("AUC", "Balanced accuracy"):
        ax.set_ylim(40, 100)
fig.suptitle(f"Input ablations of the Original Model (mean of {len(SEEDS)} seeds, 95% CI; ↓/↑ significant change vs. full model, Holm)", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Ablation.png"), dpi=150, bbox_inches="tight"); plt.show()
print(sorted(os.listdir(PLOT_DIR)))


['AIF360-Ethnicity.png', 'AIF360-Gender-Ethnicity.png', 'AIF360-Gender.png', 'Ablation.png', 'Data-vs-Model-Bias.png', 'Fairlearn-Ethnicity-Demographic_Parity.png', 'Fairlearn-Ethnicity-Equalized_Odds.png', 'Fairlearn-Gender-Demographic_Parity.png', 'Fairlearn-Gender-Equalized_Odds.png', 'Fairlearn-Gender-Ethnicity-Demographic_Parity.png', 'Fairlearn-Gender-Ethnicity-Equalized_Odds.png', 'Fairness-Utility.png', 'Subgroup-Sensitivity.png']
